# CPC + Return Regression Training

Two-phase training for crypto trading:
1. **Phase 1: CPC Pretraining** - Self-supervised learning on price sequences
2. **Phase 2: Return Regression** - Multi-task prediction (return + drawdown)

Uses Kelly Criterion for optimal position sizing at inference.

## 1. Setup Environment

In [ ]:
# Clone repository
!git clone https://github.com/your-repo/gmgn_trading.git || echo "Already cloned"
%cd gmgn_trading

In [ ]:
# Install dependencies
!pip install -q torch numpy pandas tqdm matplotlib scikit-learn

In [ ]:
# Check GPU and auto-configure
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")

In [ ]:
# GPU-specific configuration
GPU_CONFIGS = {
    'A100': {'batch': 512, 'hidden': 512, 'heads': 16, 'embed': 1024, 'lstm_layers': 3},
    'H100': {'batch': 512, 'hidden': 512, 'heads': 16, 'embed': 1024, 'lstm_layers': 3},
    'L4':   {'batch': 384, 'hidden': 384, 'heads': 12, 'embed': 768, 'lstm_layers': 2},
    'T4':   {'batch': 256, 'hidden': 256, 'heads': 8, 'embed': 512, 'lstm_layers': 2},
}

def detect_gpu():
    if not torch.cuda.is_available():
        return 'T4'  # Default
    name = torch.cuda.get_device_name(0).upper()
    for gpu in GPU_CONFIGS:
        if gpu in name:
            return gpu
    return 'T4'

GPU_TYPE = detect_gpu()
CONFIG = GPU_CONFIGS[GPU_TYPE]
print(f"\nConfigured for: {GPU_TYPE}")
print(f"Batch size: {CONFIG['batch']}")
print(f"Hidden dim: {CONFIG['hidden']}")
print(f"Embed dim: {CONFIG['embed']}")

## 2. Load and Verify Data

In [ ]:
import sys
sys.path.insert(0, 'ai_model/src')

import pickle
import numpy as np
from pathlib import Path

# Load preprocessed data
DATA_DIR = Path('ai_model/data/processed')

with open(DATA_DIR / 'train_samples.pkl', 'rb') as f:
    train_samples = pickle.load(f)

with open(DATA_DIR / 'val_samples.pkl', 'rb') as f:
    val_samples = pickle.load(f)

with open(DATA_DIR / 'test_samples.pkl', 'rb') as f:
    test_samples = pickle.load(f)

print(f"Training samples: {len(train_samples):,}")
print(f"Validation samples: {len(val_samples):,}")
print(f"Test samples: {len(test_samples):,}")

In [ ]:
# Analyze return distribution
import matplotlib.pyplot as plt

returns = [s['potential_profit_pct'] for s in train_samples]
drawdowns = [s['drawdown_pct'] for s in train_samples]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(returns, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(returns), color='red', linestyle='--', label=f'Mean: {np.mean(returns):.2%}')
axes[0].set_xlabel('Potential Profit %')
axes[0].set_ylabel('Count')
axes[0].set_title('Return Distribution')
axes[0].legend()

axes[1].hist(drawdowns, bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1].axvline(np.mean(drawdowns), color='blue', linestyle='--', label=f'Mean: {np.mean(drawdowns):.2%}')
axes[1].set_xlabel('Drawdown %')
axes[1].set_ylabel('Count')
axes[1].set_title('Drawdown Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nReturn stats: mean={np.mean(returns):.2%}, std={np.std(returns):.2%}")
print(f"Drawdown stats: mean={np.mean(drawdowns):.2%}, std={np.std(drawdowns):.2%}")

## 3. Phase 1: CPC Pretraining

Self-supervised learning using Contrastive Predictive Coding.
The model learns to predict future embeddings from context.

In [ ]:
from cpc_regression.config import CPCConfig, RegressionConfig, KellyConfig
from cpc_regression.encoder import CPCEncoder
from cpc_regression.cpc_model import CPCModel
from training.cpc_trainer import train_cpc

# Create CPC config with GPU-specific settings
cpc_config = CPCConfig(
    input_dim=14,
    hidden_dim=CONFIG['hidden'],
    embed_dim=CONFIG['embed'],
    lstm_layers=CONFIG['lstm_layers'],
    n_heads=CONFIG['heads'],
    ff_dim=CONFIG['hidden'] * 4,
    ar_hidden=CONFIG['hidden'],
    batch_size=CONFIG['batch'],
    learning_rate=1e-4,
    total_epochs=50,
    prediction_steps=[1, 2, 3, 5, 10],
    temperature=0.07,
)

print("CPC Configuration:")
print(f"  Encoder: {cpc_config.hidden_dim}h x {cpc_config.lstm_layers}L x {cpc_config.n_heads}H")
print(f"  Embed dim: {cpc_config.embed_dim}")
print(f"  Batch size: {cpc_config.batch_size}")
print(f"  Epochs: {cpc_config.total_epochs}")

In [ ]:
# Run CPC pretraining
OUTPUT_DIR = Path('ai_model/models/cpc_regression')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cpc_results = train_cpc(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    config=cpc_config,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=1,
)

print(f"\nCPC Pretraining Complete!")
print(f"Best validation loss: {cpc_results['best_val_loss']:.4f}")
print(f"Best validation accuracy: {cpc_results['history']['val_acc'][cpc_results['best_epoch']-1]:.2%}")

In [ ]:
# Plot CPC training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cpc_results['history']['train_loss'], label='Train')
axes[0].plot(cpc_results['history']['val_loss'], label='Validation')
axes[0].axvline(cpc_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CPC Loss')
axes[0].legend()

axes[1].plot(cpc_results['history']['train_acc'], label='Train')
axes[1].plot(cpc_results['history']['val_acc'], label='Validation')
axes[1].axvline(cpc_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CPC Prediction Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Visualize Learned Representations (Optional)

In [ ]:
# t-SNE visualization of learned embeddings
from sklearn.manifold import TSNE
from training.cpc_trainer import load_pretrained_encoder

# Load best encoder
encoder, _ = load_pretrained_encoder(
    str(OUTPUT_DIR / 'best_cpc_model.pt'),
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
encoder.eval()

# Extract embeddings for a subset of validation samples
N_SAMPLES = 1000
embeddings = []
labels = []

with torch.no_grad():
    for sample in val_samples[:N_SAMPLES]:
        features = torch.FloatTensor(sample['features']).unsqueeze(0)
        if torch.cuda.is_available():
            features = features.cuda()
        z = encoder.get_last_embedding(features)
        embeddings.append(z.cpu().numpy())
        # Label by return: 0=negative, 1=small, 2=large
        r = sample['potential_profit_pct']
        if r < 0:
            labels.append(0)
        elif r < 0.05:
            labels.append(1)
        else:
            labels.append(2)

embeddings = np.vstack(embeddings)
labels = np.array(labels)

print(f"Embeddings shape: {embeddings.shape}")

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                      c=labels, cmap='RdYlGn', alpha=0.6, s=10)
plt.colorbar(scatter, label='Return Category (0=neg, 1=small, 2=large)')
plt.title('t-SNE of CPC Embeddings')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.show()

## 5. Phase 2: Return Regression

Fine-tune the pretrained encoder for return prediction.
Multi-task learning: predict both return and drawdown.

In [ ]:
from training.regression_trainer import train_regression

# Create regression config
reg_config = RegressionConfig(
    hidden_dims=[256, 128],
    predict_drawdown=True,
    batch_size=CONFIG['batch'] // 2,  # Smaller batch for fine-tuning
    learning_rate=3e-5,
    encoder_lr_mult=0.1,
    freeze_encoder_epochs=5,
    total_epochs=30,
    drawdown_weight=0.3,
    patience=10,
)

print("Regression Configuration:")
print(f"  Batch size: {reg_config.batch_size}")
print(f"  Learning rate: {reg_config.learning_rate}")
print(f"  Encoder LR mult: {reg_config.encoder_lr_mult}")
print(f"  Freeze epochs: {reg_config.freeze_encoder_epochs}")
print(f"  Predict drawdown: {reg_config.predict_drawdown}")

In [ ]:
# Run regression training
reg_results = train_regression(
    pretrained_encoder_path=str(OUTPUT_DIR / 'best_cpc_model.pt'),
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    cpc_config=cpc_config,
    reg_config=reg_config,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=1,
)

print(f"\nRegression Training Complete!")
print(f"Best validation loss: {reg_results['best_val_loss']:.4f}")
print(f"Final MAE: {reg_results['final_val_mae']:.4f}")
print(f"Final correlation: {reg_results['final_correlation']:.3f}")

In [ ]:
# Plot regression training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(reg_results['history']['train_loss'], label='Train')
axes[0].plot(reg_results['history']['val_loss'], label='Validation')
axes[0].axvline(reg_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Regression Loss')
axes[0].legend()

axes[1].plot(reg_results['history']['val_mae'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Mean Absolute Error')

axes[2].plot(reg_results['history']['val_correlation'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Correlation')
axes[2].set_title('Prediction Correlation')

plt.tight_layout()
plt.show()

## 6. Calibration Analysis

In [ ]:
from training.regression_trainer import load_regression_model
from cpc_regression.return_head import CalibrationMetrics

# Load best model
model, configs = load_regression_model(
    str(OUTPUT_DIR / 'best_regression_model.pt'),
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# Evaluate on test set
all_mus = []
all_log_vars = []
all_targets = []
all_drawdowns = []
all_dd_preds = []

model.eval()
with torch.no_grad():
    for sample in test_samples:
        features = torch.FloatTensor(sample['features']).unsqueeze(0)
        if torch.cuda.is_available():
            features = features.cuda()
        
        outputs = model(features)
        mu, log_var, dd_pred = outputs
        
        all_mus.append(mu.cpu().item())
        all_log_vars.append(log_var.cpu().item())
        all_targets.append(sample['potential_profit_pct'])
        all_drawdowns.append(sample['drawdown_pct'])
        all_dd_preds.append(dd_pred.cpu().item())

all_mus = np.array(all_mus)
all_log_vars = np.array(all_log_vars)
all_targets = np.array(all_targets)

# Calibration metrics
cal = CalibrationMetrics.compute(
    torch.FloatTensor(all_mus),
    torch.FloatTensor(all_log_vars),
    torch.FloatTensor(all_targets)
)

print("Calibration Metrics:")
print(f"  Within 1 sigma: {cal['within_1_sigma']:.1%} (expected: 68.3%)")
print(f"  Within 2 sigma: {cal['within_2_sigma']:.1%} (expected: 95.4%)")
print(f"  Within 3 sigma: {cal['within_3_sigma']:.1%} (expected: 99.7%)")
print(f"  Calibration error: {cal['calibration_error']:.3f}")

In [ ]:
# Plot predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot
axes[0].scatter(all_targets, all_mus, alpha=0.3, s=5)
axes[0].plot([-0.5, 0.5], [-0.5, 0.5], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].set_title(f'Predicted vs Actual (corr={np.corrcoef(all_targets, all_mus)[0,1]:.3f})')
axes[0].legend()
axes[0].set_xlim(-0.3, 0.5)
axes[0].set_ylim(-0.3, 0.5)

# Uncertainty calibration
sigmas = np.exp(0.5 * all_log_vars)
z_scores = np.abs((all_targets - all_mus) / sigmas)

axes[1].hist(z_scores, bins=50, density=True, alpha=0.7, edgecolor='black')
# Expected half-normal distribution
from scipy import stats
x = np.linspace(0, 4, 100)
axes[1].plot(x, 2 * stats.norm.pdf(x), 'r-', label='Expected (half-normal)')
axes[1].axvline(1, color='green', linestyle='--', alpha=0.5, label='1 sigma')
axes[1].axvline(2, color='orange', linestyle='--', alpha=0.5, label='2 sigma')
axes[1].set_xlabel('|Z-score|')
axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Calibration')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Kelly Backtesting

In [ ]:
from cpc_regression.kelly_sizer import KellyPositionSizer, KellyBacktester

# Create Kelly sizer (Quarter Kelly - conservative)
kelly_config = KellyConfig(
    kelly_fraction=0.25,
    max_position=0.05,
    min_edge=0.02,
    max_variance=0.01,
    transaction_cost=0.007,
)

sizer = KellyPositionSizer(
    kelly_fraction=kelly_config.kelly_fraction,
    max_position=kelly_config.max_position,
    min_edge=kelly_config.min_edge,
    max_variance=kelly_config.max_variance,
    transaction_cost=kelly_config.transaction_cost,
)

# Run backtest
backtester = KellyBacktester(sizer, initial_capital=1.0)
backtest_results = backtester.run(
    mus=all_mus,
    log_vars=all_log_vars,
    actual_returns=all_targets,
)

print("\nBacktest Results (Quarter Kelly):")
print(f"  Final value: {backtest_results['final_value']:.4f}")
print(f"  Total return: {backtest_results['total_return']:.2%}")
print(f"  Total trades: {backtest_results['total_trades']}")
print(f"  Win rate: {backtest_results['win_rate']:.1%}")
print(f"  Avg trade return: {backtest_results['avg_trade_return']:.2%}")
print(f"  Max drawdown: {backtest_results['max_drawdown']:.2%}")

In [ ]:
# Compare different Kelly fractions
comparison = backtester.run_comparison(
    mus=all_mus,
    log_vars=all_log_vars,
    actual_returns=all_targets,
    kelly_fractions=[0.1, 0.25, 0.5, 1.0],
)

# Plot equity curves
plt.figure(figsize=(12, 5))
for name, results in comparison.items():
    plt.plot(results['equity_curve'], label=f"{name} (ret={results['total_return']:.1%})")

plt.xlabel('Trade')
plt.ylabel('Portfolio Value')
plt.title('Equity Curve Comparison (Different Kelly Fractions)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Summary table
print("\nKelly Fraction Comparison:")
print(f"{'Fraction':<12} {'Return':<10} {'Trades':<10} {'Win Rate':<10} {'Max DD':<10}")
print("-" * 52)
for name, results in comparison.items():
    print(f"{name:<12} {results['total_return']:>8.1%} {results['total_trades']:>10} "
          f"{results['win_rate']:>9.1%} {results['max_drawdown']:>9.1%}")

## 8. Save Final Model

In [ ]:
# Save to Google Drive (if available)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    DRIVE_PATH = Path('/content/drive/MyDrive/gmgn_models')
    DRIVE_PATH.mkdir(parents=True, exist_ok=True)
    
    # Copy best model
    shutil.copy(OUTPUT_DIR / 'best_regression_model.pt', DRIVE_PATH / 'best_regression_model.pt')
    shutil.copy(OUTPUT_DIR / 'best_cpc_model.pt', DRIVE_PATH / 'best_cpc_model.pt')
    
    print(f"Models saved to Google Drive: {DRIVE_PATH}")
except:
    print("Google Drive not available. Models saved locally.")
    print(f"Model path: {OUTPUT_DIR}")

In [ ]:
# Print model summary
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nCPC Pretraining:")
print(f"  Best epoch: {cpc_results['best_epoch']}")
print(f"  Best val loss: {cpc_results['best_val_loss']:.4f}")
print(f"  Training time: {cpc_results['total_time_seconds']/60:.1f} min")

print(f"\nRegression Fine-tuning:")
print(f"  Best epoch: {reg_results['best_epoch']}")
print(f"  Best val loss: {reg_results['best_val_loss']:.4f}")
print(f"  Final MAE: {reg_results['final_val_mae']:.4f}")
print(f"  Final correlation: {reg_results['final_correlation']:.3f}")
print(f"  Training time: {reg_results['total_time_seconds']/60:.1f} min")

print(f"\nBacktest (Quarter Kelly):")
print(f"  Total return: {backtest_results['total_return']:.2%}")
print(f"  Win rate: {backtest_results['win_rate']:.1%}")
print(f"  Max drawdown: {backtest_results['max_drawdown']:.2%}")

print(f"\nModel saved to: {OUTPUT_DIR}")